**This code contains:**
- calculation of PDFs for CeO2-based CexOy clusters
- calculation of PDFs for Ce40-based CexOy clusters:
- preparation of experimental PDFs for evaluation by the trained model

**Note:**

CSD (Cambridge Structural Database) structures require additional validation due to potential formatting issues and unreasonable G(r) values. For CSD PDF calculation with quality control, please use the dedicated utility notebook:

**`utils/cif-preparePDF.ipynb`**

This utility:
- Validates calculated PDFs for unreasonable G(r) values
- Creates exclusion lists for problematic CIF files
- Provides visualization and statistics for quality control
- Only saves PDFs that pass validation

Run the utility notebook before proceeding to model training with CSD data.

In [1]:
from diffpy.Structure import loadStructure
from diffpy.srreal.pdfcalculator import DebyePDFCalculator
from pyobjcryst import loadCrystal
import glob
import numpy as np
import pandas as pd

# Import path configuration
import sys
sys.path.insert(0, '..')  # Add parent directory to path
from config import setup_workdir, get_path

### Calculation of PDFs for CeO2-based CexOy clusters

In [2]:
# Set working directory to CeO2 model clusters
setup_workdir('ceo2_model_clusters')
ceo2_files = glob.glob('*.xyz')

Working directory: /workspace/home/pdf-nn-data/ceo2_clusters/model_clusters


In [3]:
# Calculate the correct number of atoms in each xyz file
for f in ceo2_files:
    with open(f, "r") as file:
        contents = file.readlines()
        new_first_line = str(len(contents) - 2) + "\n"
        contents[0] = new_first_line
    with open(f, "w") as file:
        file.writelines(contents)

In [4]:
# Calculate PDFs and save to ceo2_calculated_pdfs folder
ceo2_output_dir = get_path('ceo2_calculated_pdfs')
ceo2_output_dir.mkdir(parents=True, exist_ok=True)

for f in ceo2_files:
    model = loadStructure(f)
    dpc = DebyePDFCalculator()
    dpc.qmax = 11
    dpc.rmax = 20
    r, g = dpc(model, qmin=0.3)
    datagcalc = np.column_stack([r, g])
    # Save to calculated_pdfs folder with same filename
    output_file = ceo2_output_dir / f.replace('.xyz', '.dat')
    np.savetxt(output_file, datagcalc, header='r g')

print(f"Saved {len(ceo2_files)} CeO2 PDF files to {ceo2_output_dir}")

Saved 10000 CeO2 PDF files to /workspace/home/pdf-nn-data/ceo2_clusters/calculated_pdfs


### Calculation of PDFs for Ce40-based CexOy clusters

In [5]:
# Set working directory to Ce model clusters
setup_workdir('ce_model_clusters')
ce_files = glob.glob('*.xyz')

Working directory: /workspace/home/pdf-nn-data/ce_clusters/model_clusters


In [6]:
# Calculate the correct number of atoms in each xyz file
for f in ce_files:
    with open(f, "r") as file:
        contents = file.readlines()
        new_first_line = str(len(contents) - 2) + "\n"
        contents[0] = new_first_line
    with open(f, "w") as file:
        file.writelines(contents)

In [7]:
# Calculate PDFs and save to ce_calculated_pdfs folder
ce_output_dir = get_path('ce_calculated_pdfs')
ce_output_dir.mkdir(parents=True, exist_ok=True)

for f in ce_files:
    model = loadStructure(f)
    dpc = DebyePDFCalculator()
    dpc.qmax = 11
    dpc.rmax = 20
    r, g = dpc(model, qmin=0.3)
    datagcalc = np.column_stack([r, g])
    # Save to calculated_pdfs folder with same filename
    output_file = ce_output_dir / f.replace('.xyz', '.dat')
    np.savetxt(output_file, datagcalc, header='r g')

print(f"Saved {len(ce_files)} Ce PDF files to {ce_output_dir}")

Saved 10000 Ce PDF files to /workspace/home/pdf-nn-data/ce_clusters/calculated_pdfs


### Experimental PDF data
Experimental PDF data may be recorded with irregular distance steps because of the different experimental setup, therefore the interpolation is required to make the experimental PDF data fully consistent with the calculated PDFs the model was trianed on. Preparation of experimental PDFs for evaluation by the trained model:

In [2]:
# Set working directory to experimental PDFs
setup_workdir('experimental_pdfs')
files = glob.glob('*.gr')

# Output directory for processed files
output_dir = get_path('experimental_processed')
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Found {len(files)} .gr files to process")
print(f"Processed files will be saved to: {output_dir}")

Working directory: /workspace/home/pdf-nn-data/experimental_pdfs
Found 27 .gr files to process
Processed files will be saved to: /workspace/home/pdf-nn-data/experimental_pdfs/processed


In [3]:
for f in files:
    data = pd.read_csv(f, header=29, delim_whitespace=True)
    df = pd.DataFrame(data)
    data.columns = ['r', 'g(r)']
    df = df.loc[(df["r"] >= 2) & (df["r"] <= 12)]
    x = np.arange(2, 12.01, 0.01)
    df["interpolated_data"] = np.interp(x, df["r"], df["g(r)"])
    df = df.drop(columns=['g(r)'])
    df.rename(columns={"interpolated_data":'g(r)'}, inplace=True)
    # Save to processed folder
    output_file = output_dir / f.replace('.gr', '_processed.gr')
    df.to_csv(output_file, index=False, sep='\t')

print(f"Processed {len(files)} experimental PDFs to {output_dir}")

Processed 27 experimental PDFs to /workspace/home/pdf-nn-data/experimental_pdfs/processed
